# Window Sensitivity

The spectral centroid reported in `audio_processing.ipynb` is computed from frames of 1024 samples
(21.3 ms) advanced by 256 samples, which gives 46.9 Hz per bin. A reader can fairly ask whether that
resolution is enough for a note whose fundamental lies near 330 Hz, and whether the natural-to-bent
centroid ratio depends on it.

This notebook recomputes the centroid from the recordings under five configurations, changing one
thing at a time:

- **A**: window 1024, hop 256, magnitude weighting (as published)
- **B**: window 4096, hop 1024, magnitude weighting
- **C**: the frames of A zero-padded to 4096 points (finer frequency grid, same time window)
- **D**: window 4096, hop 1024, power weighting
- **E**: window 1024, hop 256, power weighting

B changes the time window and the frequency grid together, C changes only the grid, and D and E change
only the weighting, so the comparison separates the three. The notebook also writes
`data/centroid_window_variants.csv`, which the sensitivity section of `permanova.ipynb` reads.

## Setup

Essentia is not used. The centroid is recomputed from the raw samples, which is also what makes the
validation below possible.

In [ ]:
import csv
import numpy as np
import statistics as st
import wave

SR = 48000

AUDIO_DIR = '../data/audios_v2'
DESCRIPTORS = '../data/descriptors_v2.csv'
OUTPUT = '../data/centroid_window_variants.csv'

Recordings are 24 bit, which `wave` returns as raw bytes, so the samples are assembled explicitly.
This is the same reader as in `harmonic_content_analysis.ipynb`.

In [ ]:
def read_wav(path):
    with wave.open(path, 'rb') as w:
        sr, width, channels = w.getframerate(), w.getsampwidth(), w.getnchannels()
        raw = w.readframes(w.getnframes())
    if width == 3:
        b = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3).astype(np.int32)
        x = b[:, 0] | (b[:, 1] << 8) | (b[:, 2] << 16)
        x = np.where(x & 0x800000, x - 0x1000000, x).astype(np.float64) / 2 ** 23
    elif width == 2:
        x = np.frombuffer(raw, dtype='<i2').astype(np.float64) / 2 ** 15
    elif width == 4:
        x = np.frombuffer(raw, dtype='<i4').astype(np.float64) / 2 ** 31
    else:
        raise ValueError(f'unsupported sample width: {width}')
    if channels > 1:
        x = x.reshape(-1, channels).mean(axis=1)
    return sr, x


def recording_path(row):
    return (f"{AUDIO_DIR}/harp{row['harp']}_{row['material']}_"
            f"{row['method']}_{row['note']}_rep{row['rep']}.wav")


rows = list(csv.DictReader(open(DESCRIPTORS)))
for r in rows:
    r['centroid'] = float(r['centroid'])
print(f'{len(rows)} recordings')

80 recordings


## Centroid with a configurable window

The function follows the chain used for the published values: Hamming window, magnitude spectrum,
centroid over the bins of each frame, then the mean over the frames whose start lies between 2 and 5 s.
The signal is padded with N/2 zeros at both ends so that frames are centred. `nfft` larger than `n`
zero-pads each frame, and `power=True` weights the frequencies by the squared magnitude.

In [3]:
def centroid(x, n, hop, nfft=None, power=False):
    nfft = nfft or n
    window = np.hamming(n)
    padded = np.concatenate([np.zeros(n // 2), x, np.zeros(n // 2)])
    starts = np.arange(0, len(padded) - n + 1, hop)
    t = starts / SR
    selected = starts[(t >= 2.0) & (t <= 5.0)]
    frames = np.stack([padded[s:s + n] * window for s in selected])
    mag = np.abs(np.fft.rfft(frames, nfft, axis=1))
    weight = mag ** 2 if power else mag
    freq = np.fft.rfftfreq(nfft, 1 / SR)
    return float(np.mean((weight @ freq) / weight.sum(axis=1)))


CONFIGS = {
    'A': dict(n=1024, hop=256),
    'B': dict(n=4096, hop=1024),
    'C': dict(n=1024, hop=256, nfft=4096),
    'D': dict(n=4096, hop=1024, power=True),
    'E': dict(n=1024, hop=256, power=True),
}

## Pipeline validation

The centroid is recomputed here with no gain, normalisation or filtering. Configuration A must
therefore reproduce the `centroid` column of `descriptors_v2.csv`, which Essentia computed after
loading the files with `MonoLoader`. Agreement shows that the loader delivers the samples unaltered.
The same loop evaluates the other four configurations.

In [4]:
for r in rows:
    sr, x = read_wav(recording_path(r))
    assert sr == SR
    r.update({name: centroid(x, **cfg) for name, cfg in CONFIGS.items()})

relative_error = np.array([abs(r['A'] / r['centroid'] - 1) for r in rows])
print(f'configuration A against the published centroid: max relative difference {relative_error.max():.1e}')

configuration A against the published centroid: max relative difference 8.1e-08


## Results by emission method

Medians over the 40 recordings of each method. The second table gives the median change of each
recording relative to configuration A.

In [5]:
by_method = {m: [r for r in rows if r['method'] == m] for m in ('bend', 'natural')}
median = lambda g, k: st.median([r[k] for r in g])

print('median centroid (Hz)')
print(f"{'':10}" + ''.join(f'{k:>9}' for k in CONFIGS))
for method, g in by_method.items():
    print(f'{method:10}' + ''.join(f'{median(g, k):9.0f}' for k in CONFIGS))

print('\nmedian change per recording, relative to A')
print(f"{'':10}" + ''.join(f'{k + "/A":>9}' for k in 'BCDE'))
for method, g in by_method.items():
    print(f'{method:10}' + ''.join(f"{st.median([r[k] / r['A'] for r in g]):9.3f}" for k in 'BCDE'))

print('\nnatural/bend ratio of the medians')
for k in CONFIGS:
    print(f"  {k}: {median(by_method['natural'], k) / median(by_method['bend'], k):.2f}")

median centroid (Hz)
                  A        B        C        D        E
bend           2140     2609     2151      758      757
natural        6848     6988     6855     3229     3229

median change per recording, relative to A
                B/A      C/A      D/A      E/A
bend          1.199    1.001    0.370    0.370
natural       1.019    1.001    0.485    0.485

natural/bend ratio of the medians
  A: 3.20
  B: 2.68
  C: 3.19
  D: 4.26
  E: 4.26


Zero-padding (C) leaves the centroid where it was, with a median change below 0.2%, so the 47 Hz
frequency grid is not what limits it. Lengthening the window (B) raises the centroid of bent notes by
about 20% and that of natural notes by about 2%, and the ratio falls from 3.20 to 2.68. The
power-weighted centroid is the same for both window lengths (D and E), and it gives a ratio of 4.26.

The pattern fits a weighting effect and not a resolution effect. With magnitude weighting, the peak of a
stable partial grows in proportion to the window length while broadband noise grows with its square
root, so a longer window gives noise more weight, and most of all in the bent notes, which have the
lower signal-to-noise ratio. Squaring the magnitudes removes the dependence on the window. The direction
of the effect does not change under any of the five configurations.

## Export for the PERMANOVA

The sensitivity section of `permanova.ipynb` reads these values. It merges them with
`descriptors_v2.csv` by harmonica, material, method, note and repetition, so those columns are kept.

In [6]:
FIELDS = ['harp', 'material', 'method', 'note', 'rep'] + list(CONFIGS)

with open(OUTPUT, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(rows)
print(f'wrote {len(rows)} rows to {OUTPUT}')

wrote 80 rows to ../data/centroid_window_variants.csv
